# Day 04 - Input / Output

Two big ideas:

1. **Formatting the output** — f-strings, `str.format()`, `%`
2. **Reading and writing files** — `open()`, `read()`, `write()`, JSON

## 1. Formatting output — f-strings

Put an `f` (or `F`) before the quotes, then write expressions inside `{ }`.

The variables or values are placed directly inside the string.

In [1]:
year = 2016
event = 'Referendum'
print(f'Results of the {year} {event}')

Results of the 2016 Referendum


## `str.format()`

`{}` are replaced by the arguments we pass.

Here `:-9` pads the number with spaces, and `:2.2%` shows a percentage with 2 decimals.

In [2]:
yes_votes = 42_572_654
total_votes = 85_705_149
percentage = yes_votes / total_votes
print('{:-9} YES votes  {:2.2%}'.format(yes_votes, percentage))

 42572654 YES votes  49.67%


## `str()` vs `repr()`

- `str(value)` → for **humans** to read
- `repr(value)` → for the **interpreter** (adds quotes, keeps special characters)

A string has two different representations: `Hello` vs `'Hello'`.

In [3]:
s = 'Hello, world.'
print(str(s))
print(repr(s))
print(str(1/7))

x = 10 * 3.25
y = 200 * 200
s = 'The value of x is ' + repr(x) + ', and y is ' + repr(y) + '...'
print(s)

hello = 'hello, world\n'
print(repr(hello))
print(repr((x, y, ('spam', 'eggs'))))

Hello, world.
'Hello, world.'
0.14285714285714285
The value of x is 32.5, and y is 40000...
'hello, world\n'
(32.5, 40000, ('spam', 'eggs'))


## `string.Template`

Another easy way: put `$name` inside the string, then replace it with `substitute(...)`.

Very easy to use, but less control over formatting.

In [4]:
from string import Template
t = Template('Hey $name, welcome to $place!')
print(t.substitute(name='Youco', place='Oran'))

Hey Youco, welcome to Oran!


## 1.1 f-strings in detail

A format spec after `:` controls the rendering.

- `{math.pi:.3f}` → 3 decimals
- an integer after `:` → minimum **field width** (nice for tables)
- `!r` → apply `repr()` to the value
- `=` → print `expression=value` (great for debugging)

In [5]:
import math
print(f'The value of pi is approximately {math.pi:.3f}.')

The value of pi is approximately 3.142.


In [6]:
table = {'Sjoerd': 4127, 'Jack': 4098, 'Dcab': 7678}
for name, phone in table.items():
    print(f'{name:10} ==> {phone:10d}')

Sjoerd     ==>       4127
Jack       ==>       4098
Dcab       ==>       7678


In [7]:
animals = 'eels'
print(f'My hovercraft is full of {animals}.')
print(f'My hovercraft is full of {animals!r}.')

My hovercraft is full of eels.
My hovercraft is full of 'eels'.


In [8]:
bugs = 'roaches'
count = 13
area = 'living room'
print(f'Debugging {bugs=} {count=} {area=}')

Debugging bugs='roaches' count=13 area='living room'


## 1.2 `str.format()` in detail

- a number inside `{}` = **position** of the argument
- a name inside `{}` = a **named** argument
- `[key]` gives access to a **dict key**
- `**table` unpacks a dict into named arguments

In [9]:
print('We are the {} who say "{}!"'.format('knights', 'Ni'))

# position of the argument
print('{0} and {1}'.format('spam', 'eggs'))
print('{1} and {0}'.format('spam', 'eggs'))

# named arguments
print('This {food} is {adjective}.'.format(food='spam', adjective='absolutely horrible'))

# positional + named together
print('The story of {0}, {1}, and {other}.'.format('Bill', 'Manfred', other='Georg'))

table = {'Sjoerd': 4127, 'Jack': 4098, 'Dcab': 8637678}
# access dict keys with [key]
print('Jack: {0[Jack]:d}; Sjoerd: {0[Sjoerd]:d}; Dcab: {0[Dcab]:d}'.format(table))
# unpack the dict with **
print('Jack: {Jack:d}; Sjoerd: {Sjoerd:d}; Dcab: {Dcab:d}'.format(**table))

We are the knights who say "Ni!"
spam and eggs
eggs and spam
This spam is absolutely horrible.
The story of Bill, Manfred, and Georg.
Jack: 4098; Sjoerd: 4127; Dcab: 8637678
Jack: 4098; Sjoerd: 4127; Dcab: 8637678


In [10]:
# aligned table of x, x*x, x*x*x using format()
for x in range(1, 11):
    print('{0:2d} {1:3d} {2:4d}'.format(x, x*x, x*x*x))

 1   1    1
 2   4    8
 3   9   27
 4  16   64
 5  25  125
 6  36  216
 7  49  343
 8  64  512
 9  81  729
10 100 1000


## 1.3 Manual formatting

Same table, but padded **by hand** with `str.rjust()`.

- `rjust(width)` → pad on the left
- `ljust(width)` → pad on the right
- `center(width)` → center
- `zfill(width)` → pad with **zeros** (understands `+` and `-`)

These methods do NOT change the string, they return a new one.

In [11]:
for x in range(1, 11):
    print(repr(x).rjust(2), repr(x*x).rjust(3), end=' ')
    print(repr(x*x*x).rjust(4))

 1   1    1
 2   4    8
 3   9   27
 4  16   64
 5  25  125
 6  36  216
 7  49  343
 8  64  512
 9  81  729
10 100 1000


In [12]:
print('12'.zfill(5))
print('-3.14'.zfill(7))
print('3.14159265359'.zfill(5))
print('|' + 'hello'.ljust(10) + '|')
print('|' + 'hello'.center(11, '-') + '|')

# old-style % formatting
print('The value of pi is approximately %5.3f.' % math.pi)

00012
-003.14
3.14159265359
|hello     |
|---hello---|
The value of pi is approximately 3.142.


## 2. Reading and writing files

`open(filename, mode, encoding)` returns a file object.

| mode | meaning |
|---|---|
| `'r'` | read (default) |
| `'w'` | write (**overwrites** the file!) |
| `'a'` | append (add at the end) |
| `'r+'` | read + write |
| `'b'` | binary mode (add to any mode) |

Always use `with ... as ...` — the file is **closed automatically**, even if an error happens.

In [13]:
# create a file for our examples
f = open('workfile', 'w', encoding='utf-8')
f.write('This is the entire file.\n')
f.write('This is the first line of the file.\n')
f.write('Second line of the file\n')
f.close()
print('workfile created')

# with: closes automatically
with open('workfile', encoding='utf-8') as f:
    read_data = f.read()

print(f.closed)   # True -> auto-closed
print(read_data)

workfile created
True
This is the entire file.
This is the first line of the file.
Second line of the file



In [14]:
# f.read() -> whole file; at the end it returns ''
f = open('workfile', encoding='utf-8')
print(repr(f.read()))
print(repr(f.read()))
f.close()

# f.readline() -> one line (the \n stays at the end)
f = open('workfile', encoding='utf-8')
print(repr(f.readline()))
print(repr(f.readline()))
print(repr(f.readline()))
f.close()

# loop over the file, line by line
f = open('workfile', encoding='utf-8')
for line in f:
    print(line, end='')
f.close()

'This is the entire file.\nThis is the first line of the file.\nSecond line of the file\n'
''
'This is the entire file.\n'
'This is the first line of the file.\n'
'Second line of the file\n'
This is the entire file.
This is the first line of the file.
Second line of the file


In [15]:
# f.write() returns the number of characters written
f = open('workfile', 'w', encoding='utf-8')
print(f.write('This is a test\n'))   # 15

value = ('the answer', 42)
s = str(value)   # convert the tuple to a string first
print(f.write(s))   # 18
f.close()

15
18


In [16]:
# f.seek(offset, origin): move the position inside the file
# origin: 0 = start, 1 = current, 2 = end
f = open('workfile', 'wb')
f.write(b'0123456789abcdef')
f.close()

f = open('workfile', 'rb+')
f.seek(5)       # go to byte 6 (from the start)
print(f.read(1))  # b'5'
f.seek(-3, 2)   # 3 bytes before the end
print(f.read(1))  # b'd'
f.close()

b'5'
b'd'


## 2.1 JSON — save structured data

JSON = a standard format to **exchange data** between programs.

- `json.dumps(x)` → convert an object to a **JSON string**
- `json.dump(x, f)` → write it **into a file**
- `json.load(f)` → **read it back** from a file

In [17]:
import json

x = [1, 'simple', 'list']
print(json.dumps(x))

with open('workfile.json', 'w', encoding='utf-8') as f:
    json.dump(x, f)       # save into a file

with open('workfile.json', encoding='utf-8') as f:
    y = json.load(f)      # read it back
print(y)

data = {'name': 'Youco', 'scores': [12, 15, 9]}
print(json.dumps(data))

[1, "simple", "list"]
[1, 'simple', 'list']
{"name": "Youco", "scores": [12, 15, 9]}


## 3. Reading a file, in practice

Most biological data lives in **text files**. We open them with `open()` and read them with methods like `.readlines()`.

First, let's create our example file `animaux.txt` (girafe / tigre / singe / souris).

In [18]:
# create the example file
with open("animaux.txt", "w") as f:
    f.write("girafe\ntigre\nsinge\nsouris\n")

# .readlines(): read ALL lines -> a list (each line keeps its \n)
filin = open("animaux.txt", "r")
print(filin.readlines())
filin.close()   # closing a file = closing the book

# after close, reading again fails
# filin.readlines()  # -> ValueError: I/O operation on closed file.

['girafe\n', 'tigre\n', 'singe\n', 'souris\n']


In [19]:
filin = open("animaux.txt", "r")
lignes = filin.readlines()      # read all lines
for ligne in lignes:
    print(ligne)                # print adds a newline + the \n -> blank lines
filin.close()

girafe

tigre

singe

souris



One empty line appears between animals because **`print()` adds its own newline** on top of the `\n` already in the file.

Same example with **`with`** — the file is closed automatically:

In [20]:
with open("animaux.txt", "r") as filin:
    lignes = filin.readlines()
    for ligne in lignes:
        print(ligne)
# outside the with block -> the file is already closed

girafe

tigre

singe

souris



## Other reading methods

- `.read()` → the **whole content** as ONE string
- `.readline()` → **one line** per call (used with a `while`)
- direct iteration → loop `for ligne in filin` (the **preferred** way)

In [21]:
# .read(): one big string
with open("animaux.txt", "r") as filin:
    print(repr(filin.read()))

'girafe\ntigre\nsinge\nsouris\n'


In [22]:
# .readline(): one line each call, stops when it returns "" (end of file)
with open("animaux.txt", "r") as filin:
    ligne = filin.readline()
    while ligne != "":
        print(ligne)
        ligne = filin.readline()

girafe

tigre

singe

souris



In [23]:
# direct iteration: the file object is "iterable"
with open("animaux.txt", "r") as filin:
    for ligne in filin:
        print(ligne)

girafe

tigre

singe

souris



## 3.2 Writing into a file

Write with `.write(text)`. Careful: if you forget the `\n`, everything lands on **one line**!

In [24]:
animaux2 = ["poisson", "abeille", "chat"]
with open("animaux2.txt", "w") as filout:
    for animal in animaux2:
        filout.write(animal)   # no \n -> everything on ONE line

with open("animaux2.txt", "r") as f:
    print(repr(f.read()))      # 'poissonabeillechat'

'poissonabeillechat'


In [25]:
# add the line break with an f-string -> one animal per line
animaux2 = ["poisson", "abeille", "chat"]
with open("animaux2.txt", "w") as filout:
    for animal in animaux2:
        filout.write(f"{animal}\n")

with open("animaux2.txt", "r") as f:
    print(f.read())

poisson
abeille
chat



## 3.3 Two files in one `with`

You can open **two files at the same time** (read one, write the other):

In [26]:
with open("animaux.txt", "r") as fichier1, open("animaux3.txt", "w") as fichier2:
    for ligne in fichier1:
        fichier2.write("* " + ligne)

with open("animaux3.txt", "r") as f:
    print(f.read())

* girafe
* tigre
* singe
* souris



## Types: file content is ALWAYS strings

Numbers inside a file come back as **strings**. Convert them with `float()` / `int()` if you want to do math.

In [27]:
with open("notes.txt", "w") as f:
    f.write("13.5\n17.0\n9.5\n12.0\n10.0\n")

notes = []
with open("notes.txt", "r") as f:
    for ligne in f:
        notes.append(float(ligne.strip()))   # strip() removes the \n

print(notes)
print(sum(notes))   # now they are real numbers

[13.5, 17.0, 9.5, 12.0, 10.0]
62.0


## Exercises

**Exercise 1 — average of the grades**: read `notes.txt`, convert to `float`, compute the mean with 2 decimals.

In [28]:
notes = []
with open("notes.txt", "r") as f:
    for ligne in f:
        notes.append(float(ligne.strip()))

moyenne = sum(notes) / len(notes)
print(f"moyenne = {moyenne:.2f}")

moyenne = 12.40


**Exercise 2 — pass or fail**: rewrite the grades into `notes2.txt`, adding `admis` (>= 10) or `recalé` (< 10), with one decimal.

In [29]:
with open("notes2.txt", "w") as f_out:
    for note in notes:
        if note >= 10.0:
            f_out.write(f"{note:.1f} admis\n")
        else:
            f_out.write(f"{note:.1f} recalé\n")

with open("notes2.txt", "r") as f:
    for ligne in f:
        print(ligne, end="")

13.5 admis
17.0 admis
9.5 recalé
12.0 admis
10.0 admis


## Exercise +++ — two-dimensional spiral

A point on a circle of radius `r`: `x = r * cos(theta)`, `y = r * sin(theta)`.

We vary both at the same time: `theta` from `0` to `4*pi` (two full turns) by `0.1`, and `r` growing from `0.5` by `0.1`.

Results are saved into `spirale.dat`.

In [30]:
import math

with open("spirale.dat", "w") as f:
    r = 0.5
    theta = 0.0
    while theta <= 4 * math.pi:
        x = r * math.cos(theta)
        y = r * math.sin(theta)
        f.write(f"{x:10.5f} {y:10.5f}\n")
        r += 0.1
        theta += 0.1

# show the first 6 lines of the file
with open("spirale.dat", "r") as f:
    for i, ligne in enumerate(f):
        if i < 6:
            print(ligne, end="")

   0.50000    0.00000
   0.59700    0.05990
   0.68605    0.13907
   0.76427    0.23642
   0.82895    0.35048
   0.87758    0.47943


In [31]:
with open("spirale.dat", "r") as f:
    print(len(f.readlines()))  # number of points

126


**Visualize it** — if you have `matplotlib` installed, this draws the spiral:

```python
import matplotlib.pyplot as plt

x = []
y = []
with open("spirale.dat", "r") as f_in:
    for line in f_in:
        coords = line.split()
        x.append(float(coords[0]))
        y.append(float(coords[1]))

fig, ax = plt.subplots(figsize=(8,8))
mini = min(x+y) - 2
maxi = max(x+y) + 2
ax.set_xlim(mini, maxi)
ax.set_ylim(mini, maxi)
ax.plot(x, y)
fig.savefig("spirale.png")
```

## Summary

| Idea | Tool | Example |
|---|---|---|
| f-string | `f"...{expr}..."` | `f'pi = {math.pi:.3f}'` |
| format() | `"...{0} {name}...".format(...)` | `'{0} and {1}'.format('a','b')` |
| human vs interpreter | `str()` vs `repr()` | `repr('hi')` → `'hi'` |
| template | `$name` + `substitute()` | `Template('Hi $name')` |
| open file | `open(name, mode, encoding)` | `open('f.txt', 'w', encoding='utf-8')` |
| auto-close | `with open(...) as f:` | reads AND closes |
| read | `f.read()`, `f.readline()` | whole file / one line |
| write | `f.write(text)` | returns number of chars |
| move | `f.seek(offset, origin)` | `f.seek(5)`, `f.seek(-3, 2)` |
| JSON string | `json.dumps(obj)` | `[1, "simple", "list"]` |
| JSON file | `json.dump(obj, f)` / `json.load(f)` | save / load |

**Reading & writing files, in practice:**

| Method | What it does |
|---|---|
| `.readlines()` | all lines as a **list** of strings |
| `.read()` | whole file as **one string** |
| `.readline()` | **one line** per call (`while`) |
| `for ligne in filin` | iterate line by line (**preferred**) |
| `.write(text)` | write text (returns number of chars) |
| `float()` / `int()` / `str()` | convert file strings back to numbers |